# S6E5 — Feature Engineering

## Setup

### Imports e configuração

In [1]:
from pathlib import Path

from mltemplate.config import ProjectConfig
from mltemplate.storage import StorageManager
from mltemplate.data import KaggleSource, DataManager
from mltemplate.features import FeatureEngineer

import logging
logging.basicConfig(level=logging.INFO)

In [2]:
config = ProjectConfig(
    target="PitNextLap",                                                                                                                                                                   # ajuste para o nome real da coluna alvo
    numerical_features=["Year", "LapNumber", "Stint", "TyreLife", "Position", "LapTime (s)", "LapTime_Delta", "Cumulative_Degradation", "RaceProgress", "Position_Change"],                # preencha após ver o dataset
    categorical_features=["Compound", "Race", "PitStop"],                                                                                                                                  # preencha após ver o dataset
    ignore_features=["id", "Driver"],                                                                                                                                                                # ajuste se necessário
    problem_type="classification",
)

storage = StorageManager(root=Path("."))
dm      = DataManager(storage, config)

### Carregar dados

In [3]:
source = KaggleSource("playground-series-s6e5")
train_df, test_df = dm.load_raw(source)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

INFO:mltemplate.data.sources:Dados já existentes em data\raw — download ignorado.


INFO:mltemplate.data.manager:Dados brutos carregados — treino (439140, 16), teste (188165, 15)


Train: (439140, 16)
Test:  (188165, 15)


## v1 — Baseline

Features numéricas brutas do dataset original (Year, LapNumber, Stint, TyreLife, Position, LapTime, LapTime_Delta, Cumulative_Degradation, RaceProgress, Position_Change) com imputação por mediana, OneHotEncoding para categóricas (Compound, Race, PitStop) e StandardScaler.

### Split treino / validação

In [4]:
X_train, X_val, y_train, y_val = dm.split(train_df)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")

INFO:mltemplate.data.manager:Split — treino (351312, 15), validação (87828, 15)


X_train: (351312, 15)
X_val:   (87828, 15)


### Pipeline de transformação

In [5]:
fe = FeatureEngineer(config)

# drop colunas ignoradas
X_train = fe.drop_ignored(X_train)
X_val   = fe.drop_ignored(X_val)
X_test  = fe.drop_ignored(test_df)

# imputação (mediana/moda aprendidas do treino)
fe.fit_imputer(X_train)
X_train = fe.impute(X_train)
X_val   = fe.impute(X_val)
X_test  = fe.impute(X_test)

# encoding categórico (fit apenas no treino)
fe.fit_encoder(X_train)
X_train = fe.encode(X_train)
X_val   = fe.encode(X_val)
X_test  = fe.encode(X_test)

# scaling numérico (fit apenas no treino)
fe.fit_scaler(X_train)
X_train = fe.scale(X_train)
X_val   = fe.scale(X_val)
X_test  = fe.scale(X_test)

drop_ignored: 2 coluna(s) removida(s) — ['id', 'Driver']
drop_ignored: 2 coluna(s) removida(s) — ['id', 'Driver']
drop_ignored: 2 coluna(s) removida(s) — ['id', 'Driver']
fit_imputer: 10 numérica(s) — medianas {'Year': '2024', 'LapNumber': '19', 'Stint': '2', 'TyreLife': '12', 'Position': '10', 'LapTime (s)': '90.52', 'LapTime_Delta': '-0.296', 'Cumulative_Degradation': '-20.99', 'RaceProgress': '0.2692', 'Position_Change': '0'}
fit_imputer: 3 categórica(s) — modas {'Compound': 'MEDIUM', 'Race': 'Dutch Grand Prix', 'PitStop': np.int64(0)}


impute: nenhum valor ausente encontrado
impute: nenhum valor ausente encontrado
impute: nenhum valor ausente encontrado
fit_encoder: 3 coluna(s) categórica(s) — ['Compound', 'Race', 'PitStop'] → 32 coluna(s) após encoding


encode: 3 coluna(s) → 32 coluna(s) gerada(s) — ['Compound_HARD', 'Compound_INTERMEDIATE', 'Compound_MEDIUM', 'Compound_SOFT', 'Compound_WET', 'Race_Abu Dhabi Grand Prix', 'Race_Australian Grand Prix', 'Race_Austrian Grand Prix', 'Race_Azerbaijan Grand Prix', 'Race_Bahrain Grand Prix', 'Race_Belgian Grand Prix', 'Race_British Grand Prix', 'Race_Canadian Grand Prix', 'Race_Chinese Grand Prix', 'Race_Dutch Grand Prix', 'Race_Emilia Romagna Grand Prix', 'Race_French Grand Prix', 'Race_Hungarian Grand Prix', 'Race_Italian Grand Prix', 'Race_Japanese Grand Prix', 'Race_Las Vegas Grand Prix', 'Race_Mexico City Grand Prix', 'Race_Miami Grand Prix', 'Race_Monaco Grand Prix', 'Race_Pre-Season Testing', 'Race_Qatar Grand Prix', 'Race_Saudi Arabian Grand Prix', 'Race_Singapore Grand Prix', 'Race_Spanish Grand Prix', 'Race_São Paulo Grand Prix', 'Race_United States Grand Prix', 'PitStop_1']
encode: 3 coluna(s) → 32 coluna(s) gerada(s) — ['Compound_HARD', 'Compound_INTERMEDIATE', 'Compound_MEDIUM', 

encode: 3 coluna(s) → 32 coluna(s) gerada(s) — ['Compound_HARD', 'Compound_INTERMEDIATE', 'Compound_MEDIUM', 'Compound_SOFT', 'Compound_WET', 'Race_Abu Dhabi Grand Prix', 'Race_Australian Grand Prix', 'Race_Austrian Grand Prix', 'Race_Azerbaijan Grand Prix', 'Race_Bahrain Grand Prix', 'Race_Belgian Grand Prix', 'Race_British Grand Prix', 'Race_Canadian Grand Prix', 'Race_Chinese Grand Prix', 'Race_Dutch Grand Prix', 'Race_Emilia Romagna Grand Prix', 'Race_French Grand Prix', 'Race_Hungarian Grand Prix', 'Race_Italian Grand Prix', 'Race_Japanese Grand Prix', 'Race_Las Vegas Grand Prix', 'Race_Mexico City Grand Prix', 'Race_Miami Grand Prix', 'Race_Monaco Grand Prix', 'Race_Pre-Season Testing', 'Race_Qatar Grand Prix', 'Race_Saudi Arabian Grand Prix', 'Race_Singapore Grand Prix', 'Race_Spanish Grand Prix', 'Race_São Paulo Grand Prix', 'Race_United States Grand Prix', 'PitStop_1']
fit_scaler: 10 coluna(s) numérica(s) — ['Year', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 

### Inspeção

In [6]:
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
X_train.head()

X_train: (351312, 42)
X_val:   (87828, 42)
X_test:  (188165, 42)


,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET,Race_Abu Dhabi Grand Prix,Race_Australian Grand Prix,Race_Austrian Grand Prix,Race_Azerbaijan Grand Prix,Race_Bahrain Grand Prix,...,Year,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
69772,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.485881,1.233221,0.223286,2.023286,0.261897,-0.472019,0.019914,0.178999,1.081485,-0.775235
78874,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.465102,0.702448,1.276594,-0.015695,1.020096,0.679107,0.542650,-0.150629,0.614734,0.473374
299442,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.510390,-0.300122,0.223286,-0.525440,1.778296,-0.316366,0.072696,0.128059,-0.331132,-0.026070
129625,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.465102,-1.007818,-0.830022,-0.831287,-0.875401,-0.059739,0.085460,-0.577094,-1.028762,1.472262
254866,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.465102,-0.005248,-0.830022,0.901847,-1.254501,0.360922,-0.174663,-0.014199,-0.167781,0.723096


### Salvar

In [7]:
dm.save_feature_set(X_train, X_test, name="v1")
print("Feature set 'v1' salvo.")

INFO:mltemplate.data.manager:Feature set 'v1' salvo em data\processed\v1


Feature set 'v1' salvo.
